# 2장 2강: 분산분석(ANOVA)과 사후 검정 이론 — 실습문제

## 실습 목표

- 세 집단 이상의 평균을 일원배치 분산분석으로 비교할 수 있다.
- F통계량과 p-value를 이용하여 전체 집단 차이를 판단할 수 있다.
- 집단 간·집단 내 변동으로 ANOVA 표를 구성하고 해석할 수 있다.
- ANOVA가 유의할 때 Tukey HSD 사후 검정을 수행할 수 있다.
- 유의한 집단 쌍과 평균 차이의 크기를 근거로 차이 구조를 설명할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing(1).csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `KitchenQual` | 주방 품질 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> 표본은 지정된 `random_state`로 추출하여 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations

data_candidates = [Path("data/ames_housing.csv"), Path("ames_housing.csv")]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("프로젝트 루트에서 실행하고 data/ames_housing.csv를 확인하세요.")
df = pd.read_csv(data_path)
alpha = 0.05
print("데이터 크기:", df.shape)
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
display(df.head())


데이터 크기: (1460, 10)
전체 결측치 수: 0
컬럼: ['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']


,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. One-way ANOVA로 전체 평균 차이 확인

### 문제 1-1. 전반적인 품질 점수 5·6·7 집단 비교

#### 문제 설명

주택의 전반적인 품질 점수가 5점, 6점, 7점인 세 집단의 평균 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출하고 일원배치 분산분석을 수행하세요.

#### 요구사항

1. `OverallQual`이 5, 6, 7인 각 집단의 `SalePrice`에서 `n=20`, `random_state=5`로 표본을 추출하세요.
2. 각 집단의 표본 수, 평균, 표준편차를 출력하세요.
3. 세 집단이 서로 다른 주택으로 구성된 독립집단임을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 세 집단에 Levene 등분산 검정을 수행하세요.
6. 다음 가설을 작성하세요.
   - H₀: 세 집단의 모집단 평균 판매가격은 모두 같다.
   - H₁: 적어도 한 집단의 모집단 평균 판매가격은 다르다.
7. `stats.f_oneway()`로 일원배치 ANOVA를 수행하세요.
8. F통계량과 p-value를 출력하고 전체 차이 유무를 판단하세요.
9. ANOVA 결과만으로 어느 집단끼리 다른지 알 수 있는지 설명하세요.

#### 해석 질문

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검 결과
- 가설 설정
- F통계량과 p-value
- 전체 차이 판단
- 사후 검정 필요성 설명
- Q1~Q4 답변

In [2]:
# 각 품질 집단은 다른 주택이며 관측치 간 독립성을 가정합니다.
quality_groups = {q: df.loc[df["OverallQual"] == q, "SalePrice"].dropna().sample(n=20, random_state=5)
                  for q in [5, 6, 7]}
summary = pd.DataFrame({q: {"n": len(g), "평균(달러)": g.mean(), "표준편차": g.std(ddof=1),
                            "Shapiro p": stats.shapiro(g).pvalue} for q, g in quality_groups.items()}).T
display(summary)
levene = stats.levene(*quality_groups.values())
print(f"Levene: statistic={levene.statistic:.6f}, p={levene.pvalue:.6g}")
print("가정 검정의 p>=0.05는 정규성·등분산성의 증명이 아니라 기각할 증거 부족을 뜻합니다.")
print("H0: 세 모집단 평균은 모두 같다. H1: 적어도 한 모집단 평균은 다르다.")
anova = stats.f_oneway(*quality_groups.values())
print(f"ANOVA: F={anova.statistic:.6f}, p={anova.pvalue:.6g}")
print("전체 평균 차이 유의; 쌍별 차이는 사후검정 필요" if anova.pvalue < alpha else "전체 차이의 증거 부족")


,n,평균(달러),표준편차,Shapiro p
5,20.0,130605.0,24937.109844,0.776001
6,20.0,167826.6,41944.554425,0.409640
7,20.0,217593.6,48298.392702,0.129004


Levene: statistic=2.651637, p=0.079227
가정 검정의 p>=0.05는 정규성·등분산성의 증명이 아니라 기각할 증거 부족을 뜻합니다.
H0: 세 모집단 평균은 모두 같다. H1: 적어도 한 모집단 평균은 다르다.
ANOVA: F=24.245575, p=2.40313e-08
전체 평균 차이 유의; 쌍별 차이는 사후검정 필요


### 필수 1 답변

- **Q1.** 여러 쌍을 각각 0.05로 검정하면 적어도 한 번 제1종 오류를 낼 확률이 커집니다. 독립인 3회 검정이라면 1−0.95³≈14.26%이나, 실제 쌍별 검정은 독립이 아니므로 이 숫자는 원리 설명용입니다.

- **Q2.** F는 **집단 간 평균제곱(MS_between) / 집단 내 평균제곱(MS_within)**입니다. 단순 제곱합의 비가 아니라 각각의 자유도로 나눈 변동의 비입니다.

- **Q3.** F=24.2456, p=2.403e-08이므로 전체 평균 동일 가설을 기각합니다. Shapiro p는 5점 0.7760, 6점 0.4096, 7점 0.1290, Levene p=0.0792로 해당 가정을 기각할 증거는 부족합니다.

- **Q4.** 알 수 없습니다. ANOVA는 적어도 하나의 평균이 다르다는 전체 검정이므로 Tukey HSD로 쌍별 차이를 확인합니다.


---

## 필수 2. ANOVA 표 구성과 Tukey HSD 사후 검정

### 문제 2-1. 품질 점수별 차이 구조 확인

#### 문제 설명

필수 1의 세 집단을 이용하여 ANOVA 표를 직접 구성하고, 전체 차이가 유의한 경우 Tukey HSD 사후 검정을 수행해 어느 품질 점수 집단끼리 차이가 있는지 확인하세요.

#### 요구사항

1. 필수 1의 세 집단을 하나의 `anova_df` 데이터프레임으로 결합하세요.
2. 전체 평균을 계산하세요.
3. 다음 값을 계산하여 ANOVA 표를 만드세요.
   - 집단 간 제곱합 `SS_between`
   - 집단 내 제곱합 `SS_within`
   - 집단 간·집단 내 자유도
   - 평균제곱 `MS_between`, `MS_within`
   - F통계량
4. 직접 계산한 F통계량이 `stats.f_oneway()` 결과와 일치하는지 확인하세요.
5. ANOVA p-value가 0.05보다 작을 때만 `pairwise_tukeyhsd()`를 실행하세요.
6. Tukey 결과에서 `reject=True`인 집단 쌍을 확인하세요.
7. 각 집단 평균을 이용하여 집단 쌍별 평균 차이를 계산하세요.
8. 어느 집단 쌍이 유의하며 차이가 가장 큰 집단 쌍은 무엇인지 해석하세요.

#### 해석 질문

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- ANOVA 표
- F통계량 대조 결과
- Tukey HSD 결과
- 유의한 집단 쌍
- 집단 쌍별 평균 차이
- Q1~Q4 답변

In [3]:
anova_df = pd.concat([pd.DataFrame({"quality": q, "price": g.to_numpy()})
                      for q, g in quality_groups.items()], ignore_index=True)
grand_mean = anova_df["price"].mean()
SS_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in quality_groups.values())
SS_within = sum(((g - g.mean())**2).sum() for g in quality_groups.values())
k, N = len(quality_groups), len(anova_df)
df_between, df_within = k - 1, N - k
MS_between, MS_within = SS_between / df_between, SS_within / df_within
F_manual = MS_between / MS_within
print(f"전체 평균: {grand_mean:,.2f}달러")
display(pd.DataFrame({"SS": [SS_between, SS_within, SS_between+SS_within],
                      "df": [df_between, df_within, N-1],
                      "MS": [MS_between, MS_within, np.nan],
                      "F": [F_manual, np.nan, np.nan]}, index=["집단 간", "집단 내", "전체"]))
print("수동 F와 scipy F 일치:", np.isclose(F_manual, anova.statistic))
assert np.isclose(F_manual, anova.statistic)
assert np.isclose(SS_between+SS_within, ((anova_df.price-grand_mean)**2).sum())
if anova.pvalue < alpha:
    tukey = pairwise_tukeyhsd(anova_df.price, anova_df.quality, alpha=alpha)
    print(tukey.summary())
    print("reject=True인 쌍:")
    for row in tukey.summary().data[1:]:
        if row[-1]:
            print(row[0], row[1])
pair_differences = pd.DataFrame([{"집단1": a, "집단2": b,
    "평균 차이(집단2-집단1, 달러)": quality_groups[b].mean()-quality_groups[a].mean()}
    for a,b in combinations(quality_groups,2)])
display(pair_differences)


전체 평균: 172,008.40달러


,SS,df,MS,F
집단 간,7.619479e+10,2,3.809739e+10,24.245575
집단 내,8.956486e+10,57,1.571313e+09,NaN
전체,1.657596e+11,59,NaN,NaN


수동 F와 scipy F 일치: True
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2 meandiff p-adj    lower       upper    reject
-----------------------------------------------------------
     5      6  37221.6  0.012  7056.6586  67386.5414   True
     5      7  86988.6    0.0 56823.6586 117153.5414   True
     6      7  49767.0 0.0006 19602.0586  79931.9414   True
-----------------------------------------------------------
reject=True인 쌍:
5 6
5 7
6 7


,집단1,집단2,"평균 차이(집단2-집단1, 달러)"
0,5,6,37221.6
1,5,7,86988.6
2,6,7,49767.0


### 필수 2 답변

- **Q1.** SS_between은 집단 평균과 전체 평균 간 거리의 제곱에 집단 크기를 곱해 합한 값이고, SS_within은 각 관측치와 소속 집단 평균 간 편차제곱의 합입니다.

- **Q2.** 동시 다중비교를 고려한 유의수준 0.05에서 그 쌍의 모집단 평균이 같다는 가설을 기각한다는 뜻입니다.

- **Q3.** 유의한 집단 쌍은 **5–6, 5–7, 6–7**입니다. 표의 meandiff는 group2−group1이고, 신뢰구간과 보정 p값도 함께 확인합니다.

- **Q4.** 5점–7점의 차이가 가장 크며, 7점 평균이 5점보다 **86,988.60달러** 높습니다.


---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질 `Ex`·`Gd`·`TA` 집단 비교

#### 문제 설명

주방 품질이 `Ex`, `Gd`, `TA`인 세 집단의 평균 판매가격을 비교합니다. 각 집단에서 20개씩 표본을 추출한 뒤 ANOVA와 Tukey HSD를 순서대로 적용하세요.

> 필수 문제에서 학습한 전체 검정→사후 검정 절차를 새로운 집단 변수에 적용하는 과제입니다.

#### 요구사항

1. `KitchenQual`이 `Ex`, `Gd`, `TA`인 각 집단의 `SalePrice`에서 `n=20`, `random_state=18`로 표본을 추출하세요.
2. 세 집단의 표본 수와 평균을 출력하세요.
3. 정규성과 등분산성을 확인하세요.
4. 일원배치 ANOVA를 수행하고 F통계량과 p-value를 출력하세요.
5. ANOVA가 유의한 경우에만 Tukey HSD 사후 검정을 수행하세요.
6. Tukey 결과에서 유의한 집단 쌍을 확인하세요.
7. 각 집단 쌍의 평균 판매가격 차이를 계산하세요.
8. 어느 집단 쌍의 차이가 가장 큰지 포함하여 주방 품질별 차이 구조를 해석하세요.

#### 해석 질문

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  
**Q2.** 사후 검정은 어떤 조건에서 수행하나요?  
**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검
- ANOVA 결과
- Tukey HSD 결과
- 유의한 집단 쌍과 평균 차이
- 최종 해석
- Q1~Q4 답변

In [4]:
kitchen_groups = {q: df.loc[df["KitchenQual"] == q, "SalePrice"].dropna().sample(n=20, random_state=18)
                  for q in ["Ex", "Gd", "TA"]}
display(pd.DataFrame({q: {"n": len(g), "평균(달러)": g.mean(), "표준편차": g.std(ddof=1),
    "Shapiro p": stats.shapiro(g).pvalue} for q,g in kitchen_groups.items()}).T)
kitchen_levene = stats.levene(*kitchen_groups.values())
print("Levene:", kitchen_levene)
print("H0: Ex, Gd, TA의 모집단 평균은 같다. H1: 적어도 하나는 다르다.")
kitchen_anova = stats.f_oneway(*kitchen_groups.values())
print(f"ANOVA: F={kitchen_anova.statistic:.6f}, p={kitchen_anova.pvalue:.6g}")
kitchen_df = pd.concat([pd.DataFrame({"quality": q, "price": g.to_numpy()})
                       for q,g in kitchen_groups.items()], ignore_index=True)
if kitchen_anova.pvalue < alpha:
    kitchen_tukey = pairwise_tukeyhsd(kitchen_df.price, kitchen_df.quality, alpha=alpha)
    print(kitchen_tukey.summary())
    print("유의한 집단 쌍:", [(r[0],r[1]) for r in kitchen_tukey.summary().data[1:] if r[-1]])
display(pd.DataFrame([{"집단1": a, "집단2": b, "평균 차이(집단2-집단1, 달러)": kitchen_groups[b].mean()-kitchen_groups[a].mean()}
                      for a,b in combinations(kitchen_groups,2)]))


,n,평균(달러),표준편차,Shapiro p
Ex,20.0,313983.05,71649.325694,0.687541
Gd,20.0,188835.00,53455.690210,0.802395
TA,20.0,137486.60,32133.069564,0.397341


Levene: LeveneResult(statistic=2.5549876323347616, pvalue=0.08656348497808213)
H0: Ex, Gd, TA의 모집단 평균은 같다. H1: 적어도 하나는 다르다.
ANOVA: F=54.799970, p=5.30518e-14
      Multiple Comparison of Means - Tukey HSD, FWER=0.05       
group1 group2  meandiff  p-adj     lower        upper     reject
----------------------------------------------------------------
    Ex     Gd -125148.05    0.0 -166883.2108  -83412.8892   True
    Ex     TA -176496.45    0.0 -218231.6108 -134761.2892   True
    Gd     TA   -51348.4 0.0122  -93083.5608   -9613.2392   True
----------------------------------------------------------------
유의한 집단 쌍: [('Ex', 'Gd'), ('Ex', 'TA'), ('Gd', 'TA')]


,집단1,집단2,"평균 차이(집단2-집단1, 달러)"
0,Ex,Gd,-125148.05
1,Ex,TA,-176496.45
2,Gd,TA,-51348.40


### 과제 답변

- **Q1.** F=54.8000, p=5.305e-14로 전체적으로 유의합니다. Shapiro p는 Ex 0.6875, Gd 0.8024, TA 0.3973, Levene p=0.0866입니다. 독립성은 표집 구조로 가정하며 이 검정들로 확인되지 않습니다.

- **Q2.** 이번 실습에서는 ANOVA p<0.05일 때 Tukey HSD를 수행합니다. 고전적 ANOVA와 Tukey는 집단 내 정규성 및 등분산 가정을 함께 고려해야 합니다.

- **Q3.** 유의한 집단 쌍은 **Ex–Gd, Ex–TA, Gd–TA**입니다.

- **Q4.** Ex–TA 차이가 가장 크고 Ex 평균이 TA보다 **176,496.45달러** 높습니다. 주방 품질에 따른 관측상 가격 차이이며 다른 주택 특성을 통제한 인과효과는 아닙니다.


---

## 실습 마무리

1. 세 집단 이상을 t검정으로 반복 비교하면 왜 제1종 오류가 커지나요?
2. ANOVA의 귀무가설과 대립가설은 무엇인가요?
3. F통계량이 크다는 것은 무엇을 의미하나요?
4. ANOVA가 유의하더라도 사후 검정이 필요한 이유는 무엇인가요?
5. Tukey HSD 결과에서 어떤 항목을 확인해야 하나요?
### 마무리 답변

1. 비교 횟수가 많아질수록 하나 이상의 잘못된 기각이 발생할 기회가 늘어납니다.

2. H0는 모든 모집단 평균 동일, H1은 적어도 하나의 평균이 다름입니다. 모든 쌍이 다르다는 뜻은 아닙니다.

3. 집단 내 변동에 비해 집단 평균 간 변동이 크다는 뜻이며, 유의성은 자유도를 반영한 p값으로 판단합니다.

4. 전체 검정으로는 어느 쌍이 다른지 식별되지 않기 때문입니다.

5. group1, group2, 평균 차이 방향, 보정 p값(p-adj), 동시 신뢰구간(lower, upper), reject를 확인합니다.
